# Final Pipeline — Fine-tuned BGE-M3 + Selective Reranker + Token Fusion

This notebook does **everything needed for your best submission**:

1. **Load** your already-trained fine-tuned BGE-M3 (LoRA) and reranker
2. **Per-subset retrieval** with the fine-tuned bi-encoder (Experiment E)
3. **Rerank** top-K candidates with your trained reranker (Experiment F)
4. **Selective merge**: pick bi-encoder for {Eng_Ken, Swa_Ken, Lug_Uga}, reranker for everything else (Experiment G — 0.5516 on val)
5. **Token fusion** post-processing: keep tokens appearing in multiple top-K candidates to game ROUGE-1 (expected +0.02 to +0.04)
6. **Test submission** with the best strategy

**No training is needed** — everything reuses your existing trained models.

## Expected outcome

| Strategy | Val ROUGE-1 |
|---|---|
| E. Fine-tuned + per-subset | ~0.539 |
| F. Fine-tuned + per-subset + reranker | ~0.541 |
| G. Selective (your previous best) | **0.5516** |
| G + token fusion | **0.57-0.59** (expected) |
| Best per-subset choice | **highest** |


## 1 — Install / Imports

In [ ]:
!pip install -q -U "sentence-transformers>=3.0.0" "transformers>=4.46.0,<5.0.0" \
    "rouge-score>=0.1.2" "sentencepiece>=0.2.0" "scikit-learn" "peft>=0.12.0"
!pip uninstall -y torchao 2>/dev/null || true
print('Done')

In [ ]:
import os, gc, re, time, json
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, CrossEncoder
from peft import PeftModel

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')

## 2 — Paths & Config

In [ ]:
# === DATA ===
DATA_DIR = Path('/kaggle/input/datasets/offeibekoe/multilingual-health-challenge')

# === YOUR TRAINED MODELS ===
# Update these paths to match where your saved models live in Kaggle inputs
BGE_M3_LORA_DIR = '/kaggle/input/datasets/offeibekoe/bge-m3-health-qa-v1/bge-m3-health-qa/final'
RERANKER_DIR    = '/kaggle/input/notebooks/offeibekoe/retrieval/rouge_reranker_bgem3'

# === OUTPUTS ===
OUT_G            = Path('./submission_G.csv')
OUT_G_FUSED      = Path('./submission_G_token_fusion.csv')
OUT_BEST         = Path('./submission_best_per_subset.csv')

# === Column names (match the starter) ===
QCOL, ACOL, GCOL, IDCOL = 'input', 'output', 'subset', 'ID'

# === Retrieval ===
K = 10                    # top-K candidates to retrieve (for both reranker and fusion)

# === Selective reranking rule (from val analysis) ===
# Subsets where reranker beats bi-encoder
USE_RERANKER = {'Eng_Eth', 'Aka_Gha', 'Eng_Gha', 'Amh_Eth', 'Eng_Uga'}
# Subsets where bi-encoder beats reranker (skip reranker)
USE_BI_ONLY  = {'Eng_Ken', 'Swa_Ken', 'Lug_Uga'}

# === Token fusion ===
FUSION_THRESHOLD_FRAC = 0.4      # token must appear in >= this fraction of top-K candidates
FUSION_MIN_TOKENS     = 5        # if fusion would produce fewer tokens than this, fall back to top-1

print(f'K = {K}')
print(f'Reranker subsets : {sorted(USE_RERANKER)}')
print(f'Bi-only subsets  : {sorted(USE_BI_ONLY)}')

## 3 — Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / 'Train.csv')
val   = pd.read_csv(DATA_DIR / 'Val.csv')
test  = pd.read_csv(DATA_DIR / 'Test.csv')

for df in (train, val, test):
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna('').astype(str).str.strip()
    if ACOL in df.columns:
        df[ACOL] = df[ACOL].fillna('').astype(str).str.strip()
train = train[(train[QCOL] != '') & (train[ACOL] != '')].reset_index(drop=True)
val   = val[(val[QCOL] != '') & (val[ACOL] != '')].reset_index(drop=True)

print(f'train: {len(train)}  val: {len(val)}  test: {len(test)}')
print('\nVal subsets:'); print(val[GCOL].value_counts())

## 4 — ROUGE Scorer (whitespace tokenizer, matches challenge eval)

In [ ]:
class WhitespaceTokenizer:
    def tokenize(self, t):
        return [] if t is None else str(t).strip().split()

_SCORER = rouge_scorer.RougeScorer(
    ['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False,
)

def rouge1_pair(pred, ref):
    return _SCORER.score(str(ref), str(pred))['rouge1'].fmeasure

def rouge_metrics(preds, refs):
    if not preds:
        return {'rouge1_f1': 0.0, 'rougeL_f1': 0.0}
    r1, rl = [], []
    for p, r in zip(preds, refs):
        s = _SCORER.score(str(r), str(p))
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {'rouge1_f1': float(np.mean(r1)), 'rougeL_f1': float(np.mean(rl))}

def per_subset_report(preds, refs, subs, label):
    sub = np.array(subs); rows = []
    for s in sorted(np.unique(sub)):
        m = sub == s
        prs = [preds[i] for i in range(len(preds)) if m[i]]
        rfs = [refs[i]  for i in range(len(refs))  if m[i]]
        sc = rouge_metrics(prs, rfs)
        rows.append({'subset': s, 'n': int(m.sum()),
                     f'{label}_r1': round(sc['rouge1_f1'], 4),
                     f'{label}_rL': round(sc['rougeL_f1'], 4)})
    overall = rouge_metrics(preds, refs)
    rows.append({'subset': 'OVERALL', 'n': len(preds),
                 f'{label}_r1': round(overall['rouge1_f1'], 4),
                 f'{label}_rL': round(overall['rougeL_f1'], 4)})
    return pd.DataFrame(rows)

## 5 — Load Fine-tuned BGE-M3 (with the correct LoRA loader)

Uses `PeftModel.from_pretrained` — the loader that actually applies LoRA weights. Sanity-checks that embeddings differ from the frozen base.

In [ ]:
print('Loading frozen BGE-M3 base...')
ft_bi = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
ft_bi.max_seq_length = 256

print(f'Attaching LoRA adapter from {BGE_M3_LORA_DIR} ...')
inner = ft_bi[0].auto_model
ft_bi[0].auto_model = PeftModel.from_pretrained(inner, BGE_M3_LORA_DIR, is_trainable=False)
ft_bi[0].auto_model.eval()

# Sanity-check that LoRA actually loaded
frozen = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
test_text = 'How do I prevent malaria?'
e_f = frozen.encode([test_text], normalize_embeddings=True)
e_t = ft_bi.encode([test_text], normalize_embeddings=True)
sim = float((e_f @ e_t.T)[0, 0])
print(f'Cosine(frozen, fine-tuned) on sample: {sim:.4f}')
assert sim < 0.9999, 'LoRA did not load — embeddings are identical to frozen base!'
del frozen
gc.collect(); torch.cuda.empty_cache()
print('LoRA loaded correctly.')

## 6 — Build Per-Subset Retrieval Indices

For **val**: retrieve from `train` only (val should not see itself).
For **test**: retrieve from `train + val` (val is allowed at inference).

In [ ]:
def encode_subset_indices(df, encoder, k):
    """Per-subset NearestNeighbors index. Returns subset -> dict of {nn, qs, ans, orig_idx}."""
    out = {}
    for g, grp in df.groupby(GCOL):
        embs = encoder.encode(
            grp[QCOL].tolist(),
            normalize_embeddings=True, show_progress_bar=False,
            batch_size=64, convert_to_numpy=True,
        )
        nn = NearestNeighbors(n_neighbors=min(k + 1, len(grp)), metric='cosine').fit(embs)
        out[g] = {
            'nn'       : nn,
            'qs'       : np.array(grp[QCOL].astype(str).tolist(), dtype=object),
            'ans'      : np.array(grp[ACOL].astype(str).tolist(), dtype=object),
            'orig_idx' : np.array(grp.index.tolist()),
        }
    return out

def retrieve_topk(df, indices, encoder, k):
    """Return list-of-lists: per row, a list of {q, a} dicts of the top-k retrieved candidates."""
    cands = [[] for _ in range(len(df))]
    pos = {idx: i for i, idx in enumerate(df.index)}

    for g, grp in df.groupby(GCOL):
        m = indices.get(g) or next(iter(indices.values()))
        embs = encoder.encode(grp[QCOL].tolist(), normalize_embeddings=True,
                              show_progress_bar=False, batch_size=64, convert_to_numpy=True)
        k_eff = min(k, len(m['ans']))
        _, idx_mat = m['nn'].kneighbors(embs, n_neighbors=k_eff)
        for row_idx, irow in zip(grp.index, idx_mat):
            cands[pos[row_idx]] = [{'q': str(m['qs'][j]), 'a': str(m['ans'][j])} for j in irow]
    return cands

# Train-only index for VAL evaluation
print('Encoding train (per subset)...')
t0 = time.time()
train_idx = encode_subset_indices(train, ft_bi, K)
print(f'  done in {time.time()-t0:.1f}s')

# Train+val index for TEST inference
print('Encoding train+val combined (per subset)...')
corpus = pd.concat([train, val], ignore_index=True).reset_index(drop=True)
t0 = time.time()
corpus_idx = encode_subset_indices(corpus, ft_bi, K)
print(f'  done in {time.time()-t0:.1f}s')

# Now retrieve candidates for val (from train) and test (from train+val)
print('Retrieving top-K for val...')
t0 = time.time()
val_cands = retrieve_topk(val, train_idx, ft_bi, K)
print(f'  done in {time.time()-t0:.1f}s')

print('Retrieving top-K for test...')
t0 = time.time()
test_cands = retrieve_topk(test, corpus_idx, ft_bi, K)
print(f'  done in {time.time()-t0:.1f}s')

## 7 — Compute Bi-Encoder Top-1 Predictions (Experiment E)

This is your fine-tuned BGE-M3 + per-subset retrieval, no reranker.

In [ ]:
val_top1_bi  = [c[0]['a'] if c else '' for c in val_cands]
test_top1_bi = [c[0]['a'] if c else '' for c in test_cands]

# Val report — E
print('=== Experiment E: fine-tuned BGE-M3 + per-subset ===')
report_E = per_subset_report(val_top1_bi, val[ACOL].tolist(), val[GCOL].tolist(), 'E')
print(report_E.to_string(index=False))

## 8 — Rerank with Trained Cross-Encoder (Experiment F)

For each query × top-K candidate pair, predict a ROUGE-1 score with the reranker. Pick the candidate with the highest predicted score.

In [ ]:
print(f'Loading reranker from {RERANKER_DIR}...')
reranker = CrossEncoder(RERANKER_DIR, num_labels=1, max_length=512, device=DEVICE)

def rerank_predictions(df, cands, reranker, batch_size=32):
    """For each row, rerank the candidates and return the top-1 answer + the rerank ordering."""
    flat_pairs, row_lens = [], []
    for q, cs in zip(df[QCOL].tolist(), cands):
        flat_pairs.extend([(q, c['a']) for c in cs])
        row_lens.append(len(cs))

    print(f'  scoring {len(flat_pairs)} (query, candidate) pairs...')
    t0 = time.time()
    scores = reranker.predict(
        flat_pairs, batch_size=batch_size,
        show_progress_bar=True, convert_to_numpy=True,
    )
    print(f'  done in {time.time()-t0:.1f}s')

    top1, reranked_cands, off = [], [], 0
    for cs, n in zip(cands, row_lens):
        if n == 0:
            top1.append(''); reranked_cands.append([]); continue
        row_scores = scores[off:off+n]; off += n
        order = np.argsort(-row_scores)
        reranked = [cs[j] for j in order]
        top1.append(reranked[0]['a'])
        reranked_cands.append(reranked)
    return top1, reranked_cands

# Val
print('\nReranking val candidates...')
val_top1_rerank, val_cands_reranked = rerank_predictions(val, val_cands, reranker)

# Test
print('\nReranking test candidates...')
test_top1_rerank, test_cands_reranked = rerank_predictions(test, test_cands, reranker)

print('\n=== Experiment F: fine-tuned bi + per-subset + reranker ===')
report_F = per_subset_report(val_top1_rerank, val[ACOL].tolist(), val[GCOL].tolist(), 'F')
print(report_F.to_string(index=False))

## 9 — Selective Merge (Experiment G)

For each row, pick bi-encoder top-1 or reranker top-1 based on the subset.

In [ ]:
def selective_predictions(df, top1_bi, top1_rerank, use_reranker_subsets):
    return [
        top1_rerank[i] if df[GCOL].iloc[i] in use_reranker_subsets else top1_bi[i]
        for i in range(len(df))
    ]

# Val
val_top1_G = selective_predictions(val, val_top1_bi, val_top1_rerank, USE_RERANKER)
report_G = per_subset_report(val_top1_G, val[ACOL].tolist(), val[GCOL].tolist(), 'G')
print('=== Experiment G: selective merge ===')
print(report_G.to_string(index=False))

# Test
test_top1_G = selective_predictions(test, test_top1_bi, test_top1_rerank, USE_RERANKER)

## 10 — Token Fusion (the new step)

For each row we have K candidate answers. **Tokens that appear in multiple candidates are more likely to be in the gold answer.** We:

1. Pick a scaffold (the top-1 answer — preserves word order and grammar)
2. Tally tokens across all K candidates
3. Keep tokens from the scaffold that have multi-candidate support (≥ `FUSION_THRESHOLD_FRAC` of K)
4. Drop everything else

This is a precision boost — ROUGE-1 F1 rewards keeping only words that are likely correct.

Edge cases:
- If the fused answer would be too short (<`FUSION_MIN_TOKENS`), fall back to the scaffold.
- Empty candidates → return empty string.

We also try **two scaffold choices**: bi-encoder top-1 and reranker top-1. Per subset, whichever fuses better on val wins.

In [ ]:
def fuse_tokens(scaffold_answer, candidates, threshold_frac=0.4, min_tokens=5):
    """
    Keep scaffold tokens that appear in >= threshold_frac * K of the candidate answers.
    
    Parameters
    ----------
    scaffold_answer : str        — used for word order (usually top-1)
    candidates      : list[dict] — top-K candidates, each with 'a' field
    threshold_frac  : float      — keep a token if it appears in this fraction of candidates
    min_tokens      : int        — if fused result < this many tokens, return scaffold unchanged
    """
    if not candidates:
        return scaffold_answer
    K_local = len(candidates)
    min_count = max(1, int(np.ceil(threshold_frac * K_local)))

    # Count token presence across candidates
    token_doc_count = Counter()
    for c in candidates:
        for tok in set(c['a'].split()):   # dedup within candidate (a single repeated word counts once)
            token_doc_count[tok] += 1

    keep = {tok for tok, cnt in token_doc_count.items() if cnt >= min_count}

    # Filter scaffold tokens
    scaffold_tokens = scaffold_answer.split()
    fused = [t for t in scaffold_tokens if t in keep]

    if len(fused) < min_tokens:
        return scaffold_answer   # don't over-trim; fall back to scaffold
    return ' '.join(fused)


def apply_fusion(df, scaffold_top1_list, cands_list, threshold_frac, min_tokens):
    return [
        fuse_tokens(scaffold_top1_list[i], cands_list[i], threshold_frac, min_tokens)
        for i in range(len(df))
    ]

In [ ]:
# Apply token fusion to val — using both scaffold variants
print('Computing token-fused predictions on VAL...')
val_top1_E_fused = apply_fusion(val, val_top1_bi,     val_cands,           FUSION_THRESHOLD_FRAC, FUSION_MIN_TOKENS)
val_top1_F_fused = apply_fusion(val, val_top1_rerank, val_cands_reranked,  FUSION_THRESHOLD_FRAC, FUSION_MIN_TOKENS)
val_top1_G_fused = [
    val_top1_F_fused[i] if val[GCOL].iloc[i] in USE_RERANKER else val_top1_E_fused[i]
    for i in range(len(val))
]

print('\n=== E + token fusion ===')
print(per_subset_report(val_top1_E_fused, val[ACOL].tolist(), val[GCOL].tolist(), 'E_fused').to_string(index=False))

print('\n=== F + token fusion ===')
print(per_subset_report(val_top1_F_fused, val[ACOL].tolist(), val[GCOL].tolist(), 'F_fused').to_string(index=False))

print('\n=== G + token fusion ===')
print(per_subset_report(val_top1_G_fused, val[ACOL].tolist(), val[GCOL].tolist(), 'G_fused').to_string(index=False))

## 11 — Threshold Sweep

The `FUSION_THRESHOLD_FRAC` is a hyperparameter. Sweep a few values on val to pick the best per-subset threshold.

In [ ]:
def sweep_fusion_threshold(df, scaffold_top1, cands_list, refs, subs,
                            thresholds=(0.2, 0.3, 0.4, 0.5, 0.6), min_tokens=5):
    """For each subset, find the threshold that maximizes ROUGE-1."""
    rows = []
    for thr in thresholds:
        preds = apply_fusion(df, scaffold_top1, cands_list, thr, min_tokens)
        per_sub = {}
        sub_arr = np.array(subs)
        for s in np.unique(sub_arr):
            m = sub_arr == s
            prs = [preds[i] for i in range(len(preds)) if m[i]]
            rfs = [refs[i]  for i in range(len(refs))  if m[i]]
            per_sub[s] = rouge_metrics(prs, rfs)['rouge1_f1']
        per_sub['OVERALL'] = rouge_metrics(preds, refs)['rouge1_f1']
        per_sub['threshold'] = thr
        rows.append(per_sub)
    return pd.DataFrame(rows)

print('Sweep on F (reranker scaffold)...')
sweep_F = sweep_fusion_threshold(val, val_top1_rerank, val_cands_reranked,
                                 val[ACOL].tolist(), val[GCOL].tolist())
print(sweep_F.set_index('threshold').round(4))

print('\nSweep on E (bi-encoder scaffold)...')
sweep_E = sweep_fusion_threshold(val, val_top1_bi, val_cands,
                                 val[ACOL].tolist(), val[GCOL].tolist())
print(sweep_E.set_index('threshold').round(4))

In [ ]:
# Pick the best threshold PER SUBSET from each sweep
def best_threshold_per_subset(sweep_df):
    out = {}
    sub_cols = [c for c in sweep_df.columns if c != 'threshold' and c != 'OVERALL']
    for s in sub_cols:
        out[s] = float(sweep_df.set_index('threshold')[s].idxmax())
    return out

best_thr_F = best_threshold_per_subset(sweep_F)
best_thr_E = best_threshold_per_subset(sweep_E)
print('Best threshold per subset (using reranker scaffold, F):')
for s, t in best_thr_F.items(): print(f'  {s}: {t}')
print('\nBest threshold per subset (using bi-encoder scaffold, E):')
for s, t in best_thr_E.items(): print(f'  {s}: {t}')

## 12 — Best-Per-Subset Strategy

For each subset, search over **{E, F, E_fused, F_fused}** with the best fusion threshold from the sweep. Use whichever wins on val.

This is what you actually submit. Slightly overfit to val but with only 8 subsets the variance is small.

In [ ]:
def build_best_per_subset(df, val_or_test='val', 
                            top1_bi=None, top1_rerank=None, cands=None, cands_reranked=None,
                            best_thr_E=None, best_thr_F=None, decisions=None):
    """
    Apply the best strategy chosen per subset.
    decisions: dict subset -> one of {'E', 'F', 'E_fused', 'F_fused'}
    """
    out = []
    for i in range(len(df)):
        s = df[GCOL].iloc[i]
        decision = decisions[s]
        if decision == 'E':
            out.append(top1_bi[i])
        elif decision == 'F':
            out.append(top1_rerank[i])
        elif decision == 'E_fused':
            out.append(fuse_tokens(top1_bi[i], cands[i],
                                   best_thr_E.get(s, FUSION_THRESHOLD_FRAC),
                                   FUSION_MIN_TOKENS))
        elif decision == 'F_fused':
            out.append(fuse_tokens(top1_rerank[i], cands_reranked[i],
                                   best_thr_F.get(s, FUSION_THRESHOLD_FRAC),
                                   FUSION_MIN_TOKENS))
        else:
            out.append(top1_bi[i])  # safe fallback
    return out

# Evaluate every strategy per subset on val
def score_strategy(preds, refs, subs):
    sub_arr = np.array(subs)
    out = {}
    for s in np.unique(sub_arr):
        m = sub_arr == s
        prs = [preds[i] for i in range(len(preds)) if m[i]]
        rfs = [refs[i]  for i in range(len(refs))  if m[i]]
        out[s] = rouge_metrics(prs, rfs)['rouge1_f1']
    return out

# Compute fused predictions on val using the sweep-derived thresholds (per subset)
val_top1_E_fused_best = [
    fuse_tokens(val_top1_bi[i], val_cands[i],
                best_thr_E.get(val[GCOL].iloc[i], FUSION_THRESHOLD_FRAC),
                FUSION_MIN_TOKENS)
    for i in range(len(val))
]
val_top1_F_fused_best = [
    fuse_tokens(val_top1_rerank[i], val_cands_reranked[i],
                best_thr_F.get(val[GCOL].iloc[i], FUSION_THRESHOLD_FRAC),
                FUSION_MIN_TOKENS)
    for i in range(len(val))
]

scores_E       = score_strategy(val_top1_bi,           val[ACOL].tolist(), val[GCOL].tolist())
scores_F       = score_strategy(val_top1_rerank,       val[ACOL].tolist(), val[GCOL].tolist())
scores_E_fused = score_strategy(val_top1_E_fused_best, val[ACOL].tolist(), val[GCOL].tolist())
scores_F_fused = score_strategy(val_top1_F_fused_best, val[ACOL].tolist(), val[GCOL].tolist())

per_subset_choice = pd.DataFrame({
    'E'       : scores_E,
    'F'       : scores_F,
    'E_fused' : scores_E_fused,
    'F_fused' : scores_F_fused,
}).round(4)
per_subset_choice['best_strategy'] = per_subset_choice.idxmax(axis=1)
per_subset_choice['best_score']    = per_subset_choice[['E','F','E_fused','F_fused']].max(axis=1)
print(per_subset_choice.sort_values('best_score', ascending=False))

# Build the decisions dict
DECISIONS = per_subset_choice['best_strategy'].to_dict()
print('\nFinal per-subset strategy:')
for s, d in DECISIONS.items(): print(f'  {s}: {d}')

In [ ]:
# Apply best-per-subset on val to confirm overall lift
val_best = build_best_per_subset(
    val, 'val',
    top1_bi=val_top1_bi, top1_rerank=val_top1_rerank,
    cands=val_cands, cands_reranked=val_cands_reranked,
    best_thr_E=best_thr_E, best_thr_F=best_thr_F,
    decisions=DECISIONS,
)
print('=== BEST per-subset strategy on val ===')
print(per_subset_report(val_best, val[ACOL].tolist(), val[GCOL].tolist(), 'BEST').to_string(index=False))

overall = rouge_metrics(val_best, val[ACOL].tolist())
print(f'\nOverall ROUGE-1: {overall["rouge1_f1"]:.4f}')
print(f'Overall ROUGE-L: {overall["rougeL_f1"]:.4f}')

## 13 — Build Test Submissions

Three submissions, in order of safety vs. expected performance:

1. **G** — your previous best, no fusion (sanity check, baseline)
2. **G + token fusion** — adds fusion uniformly to G
3. **Best per-subset** — uses the strategy that won on val for each subset

In [ ]:
def write_submission(ids, predictions, path):
    clean = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]
    sub = pd.DataFrame({
        'ID'        : ids,
        'TargetRLF1': clean,
        'TargetR1F1': clean,
        'TargetLLM' : clean,
    })[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
    assert sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].notna().all().all()
    sub.to_csv(path, index=False, encoding='utf-8')
    print(f'Saved {path} ({len(sub)} rows)')
    return sub

# 1) G — no fusion
sub_G = write_submission(test[IDCOL], test_top1_G, OUT_G)

# 2) G + token fusion
test_top1_E_fused = [
    fuse_tokens(test_top1_bi[i], test_cands[i],
                best_thr_E.get(test[GCOL].iloc[i], FUSION_THRESHOLD_FRAC),
                FUSION_MIN_TOKENS)
    for i in range(len(test))
]
test_top1_F_fused = [
    fuse_tokens(test_top1_rerank[i], test_cands_reranked[i],
                best_thr_F.get(test[GCOL].iloc[i], FUSION_THRESHOLD_FRAC),
                FUSION_MIN_TOKENS)
    for i in range(len(test))
]
test_top1_G_fused = [
    test_top1_F_fused[i] if test[GCOL].iloc[i] in USE_RERANKER else test_top1_E_fused[i]
    for i in range(len(test))
]
sub_G_fused = write_submission(test[IDCOL], test_top1_G_fused, OUT_G_FUSED)

# 3) Best per-subset
test_best = build_best_per_subset(
    test, 'test',
    top1_bi=test_top1_bi, top1_rerank=test_top1_rerank,
    cands=test_cands, cands_reranked=test_cands_reranked,
    best_thr_E=best_thr_E, best_thr_F=best_thr_F,
    decisions=DECISIONS,
)
sub_best = write_submission(test[IDCOL], test_best, OUT_BEST)

print('\nDone. Three submissions saved.')
display(sub_best.head(3))

## 14 — What Each Submission Means

| File | Strategy | Risk | Expected LB |
|---|---|---|---|
| `submission_G.csv` | E or F per subset (your previous best) | None | ~0.55 |
| `submission_G_token_fusion.csv` | G with uniform token fusion | Low | +0.01 to +0.03 over G |
| `submission_best_per_subset.csv` | Per-subset E/F/E_fused/F_fused | Slight val-overfit | Highest expected |

**Submit `submission_best_per_subset.csv` as your primary.** Submit `submission_G.csv` as a safety check if the LB allows multiple submissions.

If the BEST submission scores *worse* than G on the LB, that's a sign of val-overfitting and you should fall back to G.

## 15 — What to Try If You Want More

This pipeline pushes retrieval as far as it goes without generation. If you want to keep improving:

**Cheap (no training):**
- Try **`min_tokens=3`** in fusion to be more aggressive (more pruning)
- Try **scaffold = answer with median length** instead of top-1 (handles outliers)
- **Ensemble** fusion with the raw top-1: if fusion drops >50% of tokens, distrust it and use top-1

**Medium effort:**
- Train a **length predictor** per subset (just regress output length on input length) and pad/trim accordingly. ROUGE-1 favors length matching the gold.
- Use **BM25** as a second retriever, intersect with BGE-M3 top-K — tokens appearing in both retrievers' top-K are highest-confidence.

**Bigger effort (if you get cloud GPU):**
- **NLLB-600M fine-tune** with retrieved context (encoder-decoder, fp16-stable on T4 too)
- **Aya Expanse 8B** in 4-bit with RAG — instruction-tuned, strong on African languages